# EDA: OA Damper Stuck (Experimental Dataset)

## Design decision: single season first, then check generalization

Per notebook 07's finding, baseline varies substantially by season and pooling is
unsafe — every comparison here is within-season (fault file vs. that same season's
baseline). Rather than analyzing all four seasons in parallel upfront, starting with
one season to establish whether this fault produces a checkable signal at all, then
spot-checking generalization across the others — cheaper, and consistent with this
project's practice of confirming a pattern before assuming it holds elsewhere.

**Starting with Winter_2022**, not Fall_2020 (flagged on three independent counts in
notebook 07 as structurally different: 4x longer duration, near-zero missing data,
elevated SAT) and not Summer/Fall (where the economizer already sits near its 10%
minimum regardless of fault status, per notebook 07's baseline findings — a stuck
damper has less room to show a visible effect against an already-static baseline).
Winter_2022's baseline showed the economizer genuinely active (~22% open), giving a
stuck-damper fault real room to produce a visible deviation.

## Fault details (per LBNL documentation)

Four severities, each a fixed OA damper position achieved by modifying the control
programming directly (not a real mechanical failure simulation, but a forced position):
- 5% open (half of the documented 10% minimum)
- 10% open (the documented minimum position)
- 50% open
- 100% open (fully open)

Per the control sequence: the OA/RA dampers are synchronized so their positions sum to
100% (e.g. OA at 35% means RA at 65%), and the documented minimum OA position is always
10%, even when the economizer is disabled. This means the 5% severity is *below* the
system's own documented floor — a genuinely abnormal state that shouldn't occur in
normal operation regardless of economizer status, making it a good "should be easy to
detect" test case, while 50%/100% represent the more classic "stuck open when it should
be more closed" failure mode.

## Hypothesis (before looking at any data)

- Expect `RTU_OA_DMPR_DM` to show the fault trivially (it's directly measuring the
  thing being manipulated) — the real question is whether *other* signals show a
  detectable secondary effect, which matters more for a real FDD system (you can't
  always trust the same sensor that might itself be part of the fault).
- Expect `RTU_MA_TEMP` (mixed air temperature) to respond to damper position, since a
  stuck-open damper in winter would pull in more cold outdoor air than intended,
  regardless of what the economizer logic thinks it's doing.
- Real, falsifiable prediction: does zone/supply air temperature actually deviate
  measurably, or does the RTU's SAT control loop (sequencing compressors/furnace to
  hit the 55°F setpoint) successfully compensate and mask the fault downstream? This
  would be a genuinely important finding either way — if SAT control fully compensates,
  a classifier relying only on SAT would miss this fault entirely.

In [1]:
import pandas as pd

files = {
    "baseline": "../data/raw/experimental/ERTU_Winter_2022.csv",
    "damper_005": "../data/raw/experimental/OA_damper_stuck_005_Winter_2022.csv",
    "damper_010": "../data/raw/experimental/OA_damper_stuck_010_Winter_2022.csv",
    "damper_050": "../data/raw/experimental/OA_damper_stuck_050_Winter_2022.csv",
    "damper_100": "../data/raw/experimental/OA_damper_stuck_100_Winter_2022.csv",
}

dfs = {label: pd.read_csv(fname, na_values=["NAN"]) for label, fname in files.items()}

for _label, df in dfs.items():
    df["Datetime"] = pd.to_datetime(df["Datetime"])

for _label, df in dfs.items():
    print(f"{_label}: shape={df.shape}, missing_values={df.isna().sum().sum()}")

baseline: shape=(2880, 57), missing_values=12155
damper_005: shape=(1440, 57), missing_values=6760
damper_010: shape=(1440, 57), missing_values=6188
damper_050: shape=(1440, 57), missing_values=2457
damper_100: shape=(1440, 57), missing_values=5577


## Load confirmed

Baseline: 2880 rows (matches Winter_2022's baseline from notebook 07). Each fault
file: 1440 rows (~1 day), consistent with the documented single-day faulted-run
design. `"NAN"` fix and `Datetime` conversion applied from the start this time.

Missing-value counts vary noticeably across severities (2,457-6,760) — plausibly just
reflecting each file's real occupied/unoccupied mode split (a 1-day file's exact
7am-10pm occupied window could shift the missing-value count meaningfully depending
on exactly when the recording started), not necessarily a new data-quality issue.
Not chasing this further right now — noting it, and will revisit only if it turns out
to distort a real comparison later, same as the "check what's checkable, document
what's left" standard used throughout this project.

In [2]:
severity_order = ["baseline", "damper_005", "damper_010", "damper_050", "damper_100"]
damper_cols = ["RTU_OA_DMPR_DM", "RTU_MA_TEMP", "RTU_OA_TEMP", "RTU_SA_TEMP"]

summary = pd.DataFrame({
    label: df[damper_cols].mean()
    for label, df in dfs.items()
}).T.loc[severity_order]

summary

,RTU_OA_DMPR_DM,RTU_MA_TEMP,RTU_OA_TEMP,RTU_SA_TEMP
baseline,21.836352,62.261043,48.612208,56.675473
damper_005,4.996528,64.637094,45.796479,56.492826
damper_010,9.996528,64.069171,37.628194,56.427396
damper_050,49.972222,58.110970,40.235049,57.094325
damper_100,99.965278,50.537256,41.639632,56.665242


## Finding: OA damper stuck is trivially detectable on its own sensor, but the fault
## is masked downstream by SAT control — a genuinely important result

`RTU_OA_DMPR_DM` confirms the injected fault positions almost exactly (4.997%, 9.997%,
49.972%, 99.965% vs. documented 5/10/50/100%) — trivial to detect directly.

`RTU_MA_TEMP` shows a real effect but **confounded by real outdoor-temperature
variation across the different recording days** (`RTU_OA_TEMP` ranges 37.6-48.6°F
across these five files, since each is a separate real 1-day test, not a controlled
simulation). At 50%/100% (damper wide open), MA_TEMP drops meaningfully (58.1, 50.5°F
vs baseline 62.3°F) — consistent with more cold outdoor air being pulled in. At
5%/10%, the effect is muddied by that day happening to have the coldest outdoor air of
the whole set, despite the damper being nearly closed — a real example of why raw
means across separately-recorded days need this kind of check, unlike the Simulated
dataset's single continuous run per severity.

**`RTU_SA_TEMP` stays flat across all four severities** (56.43-57.09°F vs. baseline
56.68°F) — even at 100% damper open. **This answers the hypothesis's central
question**: the RTU's supply air temperature control loop successfully compensates for
this fault, at least at the mean level. A classifier relying on `RTU_SA_TEMP` alone
would likely miss this fault entirely, regardless of severity — the fault is real and
measurable upstream (`RTU_MA_TEMP`, trivially `RTU_OA_DMPR_DM` itself) but effectively
invisible at the controlled output.

**Practical implication for modeling**: this fault needs to be detected via
`RTU_OA_DMPR_DM` or `RTU_MA_TEMP` directly — not via downstream comfort/output metrics
like SAT, which the control system actively works to normalize regardless of the
fault's presence. This is a genuinely different detection challenge than any
Simulated-dataset fault, where the faulted signal itself (capacity, pressure) was also
the thing degrading — here the *symptom* users would actually notice (SAT, comfort) is
specifically what gets hidden by design.

In [3]:
# check whether MA_TEMP tracks OA_TEMP proportionally the same way in every file
# (i.e., is the MA/OA relationship itself different, or just riding on different OA_TEMP days?)
summary["MA_minus_OA"] = summary["RTU_MA_TEMP"] - summary["RTU_OA_TEMP"]
summary

,RTU_OA_DMPR_DM,RTU_MA_TEMP,RTU_OA_TEMP,RTU_SA_TEMP,MA_minus_OA
baseline,21.836352,62.261043,48.612208,56.675473,13.648835
damper_005,4.996528,64.637094,45.796479,56.492826,18.840615
damper_010,9.996528,64.069171,37.628194,56.427396,26.440977
damper_050,49.972222,58.110970,40.235049,57.094325,17.875922
damper_100,99.965278,50.537256,41.639632,56.665242,8.897624


## Normalizing for outdoor-temperature confound reveals a clean, physically-sensible
## signal — with one small exception

`MA_TEMP - OA_TEMP` (controlling for each file's different real outdoor conditions)
shows the expected monotonic relationship for three of four severities:

| Severity | Damper position | MA - OA gap |
|---|---|---|
| damper_010 (10%) | 9.997% | 26.44°F |
| damper_050 (50%) | 49.972% | 17.88°F |
| damper_100 (100%) | 99.965% | 8.90°F |

As the damper opens further, less return air (warm) and more outdoor air (cold) enters
the mix, shrinking the gap between mixed-air and outdoor-air temperature — physically
correct, and a genuinely clean signal once the outdoor-temperature confound is removed.
Baseline (21.8% open) sits at 13.65°F, appropriately between the 010% and 050% values,
another consistency check that the relationship holds.

**`damper_005` (18.84°F) breaks the trend** — lower than `damper_010` (26.44°F), when
the most-closed damper position should show the *largest* gap, not a smaller one than
a more-open position. Not yet explained. Possible causes not yet checked: this specific
file's outdoor conditions may involve something the simple OA_TEMP mean doesn't capture
(e.g. more variable weather within that one day, or wind effects on infiltration not
represented in this dataset at all), or day-to-day operational differences unrelated to
outdoor temperature. Genuinely open, flagged rather than smoothed over — consistent
with this project's standard for honestly unresolved findings.

**Practical implication, refined**: `MA_TEMP - OA_TEMP` looks like a stronger candidate
feature than raw `MA_TEMP` for this fault, since it's far less sensitive to whichever
day happened to be recorded — worth carrying forward into feature engineering. SAT
compensation finding still holds: `RTU_SA_TEMP` stays flat regardless of severity.

## Checking generalization: does the same pattern hold in Spring_2021?

Per the design decision at the start of this notebook, checking whether Winter_2022's
findings — trivial detection via `RTU_OA_DMPR_DM`, a physically clean signal in
`MA_TEMP - OA_TEMP` (modulo the damper_005 anomaly), and full compensation in
`RTU_SA_TEMP` — hold in a second season, rather than assuming Winter_2022 generalizes.
Spring_2021 chosen since its baseline also showed the economizer genuinely active
(~22% open, per notebook 07), making it a fair comparison to Winter rather than a
season where the damper's already near its floor regardless of fault status.

In [4]:
files_spring = {
    "baseline": "../data/raw/experimental/ERTU_Spring_2021.csv",
    "damper_005": "../data/raw/experimental/OA_damper_stuck_005_Spring_2021.csv",
    "damper_010": "../data/raw/experimental/OA_damper_stuck_010_Spring_2021.csv",
    "damper_050": "../data/raw/experimental/OA_damper_stuck_050_Spring_2021.csv",
    "damper_100": "../data/raw/experimental/OA_damper_stuck_100_Spring_2021.csv",
}

dfs_spring = {label: pd.read_csv(fname, na_values=["NAN"]) for label, fname in files_spring.items()}

for _label, df in dfs_spring.items():
    df["Datetime"] = pd.to_datetime(df["Datetime"])

summary_spring = pd.DataFrame({
    label: df[damper_cols].mean()
    for label, df in dfs_spring.items()
}).T.loc[severity_order]

summary_spring["MA_minus_OA"] = summary_spring["RTU_MA_TEMP"] - summary_spring["RTU_OA_TEMP"]
summary_spring

,RTU_OA_DMPR_DM,RTU_MA_TEMP,RTU_OA_TEMP,RTU_SA_TEMP,MA_minus_OA
baseline,22.154783,62.218347,44.608271,56.092224,17.610077
damper_005,4.996528,65.906549,44.018472,56.217608,21.888076
damper_010,9.996528,66.595915,52.970937,56.157855,13.624978
damper_050,49.972222,65.669560,58.290257,55.646840,7.379303
damper_100,99.965278,62.985129,62.076000,55.940261,0.909129


## Generalization check confirms two findings, but not the damper_005 anomaly —
## a genuinely useful negative result

| Finding | Winter_2022 | Spring_2021 | Replicates? |
|---|---|---|---|
| `RTU_OA_DMPR_DM` matches documented positions | Yes | Yes | **Yes** |
| `SA_TEMP` stays flat (compensation) | 56.43-57.09°F | 55.65-56.22°F | **Yes** — both seasons show ~0.6-0.7°F range |
| `MA_minus_OA` monotonic across 010/050/100 | Yes (26.4→17.9→8.9) | Yes (13.6→7.4→0.9) | **Yes** |
| `damper_005` breaks the monotonic trend | Yes (18.8, lower than 010's 26.4) | **No** — 21.9, correctly the highest of all five | **Does not replicate** |

**This is a genuinely useful result**: two of the three real findings (trivial
detection, SAT compensation) hold up in a second season, giving real confidence
they're properties of this fault, not one-season artifacts. The `damper_005` anomaly,
however, does **not** replicate — Spring_2021 shows the textbook-correct pattern
(005% has the *largest* MA-OA gap, as physically expected for the most-closed
position), while Winter_2022 showed 005% breaking rank.

**Revised conclusion**: Winter_2022's `damper_005` anomaly was very likely a one-off —
probably driven by that specific day's weather conditions in a way the simple OA_TEMP
mean didn't capture (as speculated when first found), rather than a real property of
the 5% severity itself. Confirmed by checking a second season rather than assuming
either explanation. This is exactly why the generalization check was worth doing
before spending more effort chasing the Winter_2022 anomaly directly — a second data
point resolved it more cheaply than deeper investigation of the first would have.

**Practical implication for modeling, now more confidently stated**: `RTU_OA_DMPR_DM`
(trivial), `MA_TEMP - OA_TEMP` (physically clean, monotonic, confirmed across two
seasons), and the fact that `RTU_SA_TEMP` will NOT help detect this fault
(confirmed compensated across two seasons) — these are now real, cross-season-checked
findings, not single-season impressions.

## Summary: OA damper stuck EDA

**Design validated**: within-season comparison approach (established in notebook 07)
worked as intended — each fault file compared only against its own season's baseline,
avoiding the pooling risk that notebook 07 identified.

**Confirmed, cross-season (Winter_2022 and Spring_2021)**:
1. `RTU_OA_DMPR_DM` detects the fault trivially — matches documented forced positions
   (5/10/50/100%) almost exactly in both seasons.
2. `RTU_SA_TEMP` stays flat across all four severities in both seasons (~0.6-0.7°F
   range) — the RTU's supply-air control loop successfully compensates for this fault
   at the controlled output. **A classifier relying on SAT alone would miss this fault
   entirely, regardless of severity.**
3. `MA_TEMP - OA_TEMP` (mixed-air-minus-outdoor-air, controlling for each file's real
   day-to-day weather) shows a clean, physically-sensible monotonic relationship
   across 10/50/100% severities in both seasons — a strong candidate feature.

**One false lead, honestly resolved via the generalization check**: Winter_2022's
`damper_005` showed a rank-breaking anomaly in `MA_minus_OA`; Spring_2021 showed the
textbook-correct pattern at the same severity. Concluded this was a one-season
artifact (likely weather-related), not a real property of the 5% severity — resolved
more cheaply by checking a second season than by chasing the first season's anomaly
directly.

**Practical implication for modeling**: this fault needs detection via
`RTU_OA_DMPR_DM` and/or `MA_TEMP - OA_TEMP`, not via downstream comfort metrics like
SAT — a genuinely different detection challenge than any Simulated-dataset fault,
where the faulted signal itself was also the thing that degraded.

**Not yet checked**: Fall_2020 and Summer_2021 (both already lower priority per
notebook 07 — Fall_2020 flagged as structurally atypical, Summer_2021 expected to
have less room for a damper-position effect since the economizer already sits near
minimum). Two of four seasons confirmed is treated as sufficient generalization
evidence for this fault; revisit only if a later fault type's results conflict with
this conclusion.

**Next**: incorrect economizer setpoint (4 severities: 6°C, 8°C, 12°C, 14°C, vs. the
documented correct setpoint of 10°C) — directly related to the same OA-temperature-
dependent control logic just explored here. Real, falsifiable question carried
forward: does this fault also get masked by SAT compensation, or does it behave
differently since it's a setpoint/logic error rather than a forced physical position?